In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
import torch.nn.functional as F
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import numpy as np
from tqdm import tqdm
import torch
import os,sys
current_file = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(current_file), '..')))

<h1> Load Dataset

In [2]:
from dataset.qm9s_dataset import Multi_process_Qm9sDataset
from dataset.nist_dataset import Multi_process_NISTDataset
from dataset.collate_fn import Multi_process_batch_collate_fn
from torch.utils.data import DataLoader

/home/zjh/miniconda3/envs/mol_spectra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h1>Load Metric

In [3]:
def batch_pearsonr(y_true: torch.Tensor, y_pred: torch.Tensor):
    vx = y_true - y_true.mean(dim=1, keepdim=True)
    vy = y_pred - y_pred.mean(dim=1, keepdim=True)

    corr = (vx * vy).sum(dim=1) / (
        torch.sqrt((vx**2).sum(dim=1)) * torch.sqrt((vy**2).sum(dim=1)) + 1e-8
    )
    return corr.mean()

def batch_cos_similarity(pred_spectra, true_spectra):
    # L2 归一化（避免长度影响）
    pred_norm = F.normalize(pred_spectra, p=2, dim=1)  # [b, 512]
    true_norm = F.normalize(true_spectra, p=2, dim=1)  # [b, 512]
    # 每个样本计算余弦相似度，结果是 [b]
    cosine_sim = (pred_norm * true_norm).sum(dim=1)
    # 求平均
    avg_cosine_sim = cosine_sim.mean()
    return avg_cosine_sim

def batch_average_spearman(pred_spectra, true_spectra):
    pred_np = pred_spectra.detach().cpu().numpy()
    true_np = true_spectra.detach().cpu().numpy()

    spearman_scores = []
    for i in range(pred_np.shape[0]):
        coef, _ = spearmanr(pred_np[i], true_np[i])
        spearman_scores.append(coef)

    avg_spearman = np.mean(spearman_scores)
    return avg_spearman

<h1>Load config

In [4]:
import yaml
qm9s = "/config/qm9s_spectra_pred.yaml"
nist = "/home/zjh/final-ssmoe-mvms/config/nist_spectra_pred.yaml"
with open(nist, "r") as f:
    config = yaml.safe_load(f)
dataset = Multi_process_NISTDataset(db_path=config["test_db_path"],dict_path=config["dict_path"],config=config)
dataloader = DataLoader(dataset, batch_size=256, shuffle=False,collate_fn=Multi_process_batch_collate_fn)

<h1>Evaluate

In [6]:
from model.ssmoe_mvms import SSMoE_MVMS
from model.model_config import MolConfig

mol_config = MolConfig(mol_pretrain_pth=config["mol_pretrain_pth"],mol_dict_pth=config["mol_dict_pth"])
model = SSMoE_MVMS(config,mol_config)

"""
Load Checkpoint
"""
qm9s_ir_ckpt_dir = "/ckpt/spectra_prediction/QM9s/ir_pred.pt"
qm9s_raman_ckpt_dir = "/ckpt/spectra_prediction/QM9s/raman_pred.pt"
nist_ir_ckpt_dir = "/home/zjh/final-ssmoe-mvms/ckpt/spectra_prediction/NIST/ir_pred.pt"


batch_pear = []
batch_cos = []
batch_spear = []
mask_ratio = config["mask_ratio"]

ckpt = torch.load(nist_ir_ckpt_dir,map_location='cpu',weights_only=False)
model.load_state_dict(ckpt['model'])
model = model.eval()
model = model.cuda()
step = 0
pear = 0.0
cos = 0.0
spear = 0.0
with torch.no_grad():
    for batch_list in tqdm(dataloader,total=len(dataloader)):
        step += 1
        src_tokens,src_coord,src_distance,src_edge_type,spectra,smi = batch_list['unimol']["src_tokens"].cuda(), batch_list['unimol']["src_coord"].cuda(), batch_list['unimol']["src_distance"].cuda(), batch_list['unimol']["src_edge_type"].cuda(), batch_list['unimol']["ir"].cuda(),batch_list['unimol']["smi"]
        x,edge_index,edge_attr,batch = batch_list['pyg']["x"].cuda(), batch_list['pyg']["edge_index"].cuda(), batch_list['pyg']["edge_attr"].cuda(),batch_list['pyg'].batch.cuda()
        _,pred,spectra_loss,rec_loss = model(src_tokens, src_coord, src_distance, src_edge_type, smi, x, edge_index, edge_attr, batch, spectra, "cuda", mask_ratio,preserve_flag=[0,1,2,3])
        pred = pred.squeeze(1)
        pear += batch_pearsonr(pred,spectra.reshape(pred.shape)).item()
        cos += batch_cos_similarity(pred,spectra.reshape(pred.shape)).item()
        spear += batch_average_spearman(pred,spectra.reshape(pred.shape)).item()
    pear /= step
    cos /= step
    spear /= step
batch_pear.append(pear)
batch_cos.append(cos)
batch_spear.append(spear)
    
print("Pearson:{}, std:{}".format(np.mean(batch_pear),np.std(batch_pear)))
print("Cos:{}, std:{}".format(np.mean(batch_cos),np.std(batch_cos)))
print("Spearman:{}, std:{}".format(np.mean(batch_spear),np.std(batch_spear)))

model name: /home/zjh/final-ssmoe-mvms/model/SMILES/MoLFormer
number of SMiles_Encoder parameters: 44.38M
model name: /home/zjh/final-ssmoe-mvms/model/SMILES/Chemberta
number of SMiles_Encoder parameters: 44.10M
Loading from /home/zjh/final-ssmoe-mvms/model/graph_mvp/model_g.pth ...
GraphMVP missing_keys:[]
GraphMVP unexpeted_keys:[]
number of Unimol parameters: 47.33M
Loading pretrained weights from /home/zjh/final-ssmoe-mvms/model/unimol/unimol.pt
missing_keys:  []
unexpeted_keys:  ['lm_head.weight', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'pair2coord_proj.linear1.weight', 'pair2coord_proj.linear1.bias', 'pair2coord_proj.linear2.weight', 'pair2coord_proj.linear2.bias', 'dist_head.dense.weight', 'dist_head.dense.bias', 'dist_head.layer_norm.weight', 'dist_head.layer_norm.bias', 'dist_head.out_proj.weight', 'dist_head.out_proj.bias', 'encoder.final_head_layer_norm.weight', 'encoder.final_head_layer_norm.bias'

100%|██████████| 3/3 [00:02<00:00,  1.29it/s]

Pearson:0.8950271010398865, std:0.0
Cos:0.9118818839391073, std:0.0
Spearman:0.8731259597671013, std:0.0
